In [ ]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import pandas as pd

df = pd.read_csv(r"C:/Users/vivek/OneDrive/Document/AML Practical/Groceries_dataset.csv")

print(df.head())
print(df.columns)

   Member_number        Date   itemDescription
0           1808  21-07-2015    tropical fruit
1           2552  05-01-2015        whole milk
2           2300  19-09-2015         pip fruit
3           1187  12-12-2015  other vegetables
4           3037  01-02-2015        whole milk
Index(['Member_number', 'Date', 'itemDescription'], dtype='str')


In [ ]:
positive_items = ['milk', 'butter', 'bread', 'cheese']
negative_items = ['candy', 'soda', 'chocolate']

def create_sentiment(item):
    item = item.lower()
    if any(word in item for word in positive_items):
        return "Positive"
    elif any(word in item for word in negative_items):
        return "Negative"
    else:
        return "Neutral"
df['sentiment'] = df['itemDescription'].apply(create_sentiment)
print(df['sentiment'].value_counts())

sentiment
Neutral     30253
Positive     6107
Negative     2405
Name: count, dtype: int64


In [ ]:
positive_items = ['milk', 'butter', 'bread', 'cheese']
negative_items = ['candy', 'soda', 'chocolate']

def create_sentiment(item):
    item = str(item).lower()
    if any(word in item for word in positive_items):
        return "Positive"
    elif any(word in item for word in negative_items):
        return "Negative"
    else:
        return "Neutral"
df['sentiment'] = df['itemDescription'].apply(create_sentiment)

print("\nSentiment Distribution:")
print(df['sentiment'].value_counts())



Sentiment Distribution:
sentiment
Neutral     30253
Positive     6107
Negative     2405
Name: count, dtype: int64


In [47]:
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)   # remove URLs
    text = re.sub(r"[^a-zA-Z]", " ", text)       # remove special characters
    
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    
    return " ".join(tokens)

df['clean_text'] = df['itemDescription'].apply(preprocess_text)

print("\nSample Cleaned Text:")
print(df[['itemDescription', 'clean_text']].head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vivek\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\vivek\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!



Sample Cleaned Text:
    itemDescription      clean_text
0    tropical fruit  tropical fruit
1        whole milk      whole milk
2         pip fruit       pip fruit
3  other vegetables       vegetable
4        whole milk      whole milk


In [ ]:

# Ensure sentiment column exists

if 'sentiment' not in df.columns:
    positive_items = ['milk', 'butter', 'bread', 'cheese']
    negative_items = ['candy', 'soda', 'chocolate']

    def create_sentiment(item):
        item = str(item).lower()
        if any(word in item for word in positive_items):
            return "Positive"
        elif any(word in item for word in negative_items):
            return "Negative"
        else:
            return "Neutral"

    df['sentiment'] = df['itemDescription'].apply(create_sentiment)
# Feature Engineering
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
# Bag of Words
bow = CountVectorizer(max_features=1000)
X_bow = bow.fit_transform(df['clean_text'])

# TF-IDF
tfidf = TfidfVectorizer(max_features=1000)
X_tfidf = tfidf.fit_transform(df['clean_text'])

# Target variable
y = df['sentiment']

# Check
print("Features shape:", X_tfidf.shape)
print("Target sample:\n", y.head())

Features shape: (38765, 182)
Target sample:
 0     Neutral
1    Positive
2     Neutral
3     Neutral
4    Positive
Name: sentiment, dtype: str


In [40]:
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)


In [41]:
# Logistic Regression
lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

# Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

In [42]:
def evaluate_model(name, y_test, y_pred):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
    print(classification_report(y_test, y_pred))


In [43]:
evaluate_model("Logistic Regression", y_test, y_pred_lr)
evaluate_model("Naive Bayes", y_test, y_pred_nb)
evaluate_model("Decision Tree", y_test, y_pred_dt)


Logistic Regression
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00       481
     Neutral       1.00      1.00      1.00      6085
    Positive       1.00      1.00      1.00      1187

    accuracy                           1.00      7753
   macro avg       1.00      1.00      1.00      7753
weighted avg       1.00      1.00      1.00      7753


Naive Bayes
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00       481
     Neutral       1.00      1.00      1.00      6085
    Positive       1.00      1.00      1.00      1187

    accuracy                           1.00      7753
   macro avg       1.00      1.00      1.00      7753
weighted avg       1.00      1.00      1.00      7753


Decision Tree
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
              precision 

In [44]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes', 'Decision Tree'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ]
})

print("\nModel Comparison:")
print(results)


Model Comparison:
                 Model  Accuracy
0  Logistic Regression       1.0
1          Naive Bayes       1.0
2        Decision Tree       1.0
